# NJ Housing Market — End-to-End Analysis

**Coverage:** All 21 New Jersey counties.  
**Pipeline:** Scrape → Clean → Geocode → Segment → Pricing Analytics → Hedonic ML → Clustering → Visualize → Map → Deck.

This notebook is a thin orchestration layer. The heavy lifting lives in `src/` so the same code that powers the CLI (`python run_pipeline.py`) drives the notebook. Re-run any cell after changing config and the project rebuilds in place.

## 0 · Setup

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.io import load_config, project_path, read_any
cfg = load_config()
print('Counties tracked :', len(cfg['counties']))
print('Random seed      :', cfg['random_seed'])

## 1 · Data Collection (Phase 1)

Source chain: **Zillow → Realtor.com → Redfin** per-county failover with UA rotation, rate limiting, and tenacity retries. If all three are blocked we drop into a synthetic generator so the downstream pipeline still produces real artefacts.

In [ ]:
from src.scraper.orchestrator import collect_listings
raw_path = collect_listings(cfg, allow_synthetic=True)
raw = read_any(raw_path)
print('Raw listings :', len(raw))
raw.head()

## 2 · Cleaning (Phase 2)

- Drop outliers ($10k–$20M price band).
- Canonicalise property types into 5 buckets.
- Median-impute beds/baths/sqft with `*_was_missing` indicator columns.
- Validate ZIPs (07xxx/08xxx) and county assignment.

In [ ]:
from src.cleaning.clean import clean_listings
clean_path, qa = clean_listings(raw_path, cfg)
clean = read_any(clean_path)
print(f'Clean rows: {len(clean):,}  ·  completeness: {qa["completeness_pct"]:.1f}%')
clean['property_type'].value_counts()

## 3 · Geocoding (Phase 3)

In [ ]:
from src.geocoding.geocode import geocode_dataframe
geo_path = geocode_dataframe(clean_path, cfg, use_live=False)
geo = read_any(geo_path)
geo[['full_address', 'zip_code', 'latitude', 'longitude']].head()

## 4 · Micro-Market Segmentation (Phase 4)

County → Municipality → ZIP cluster (ZIPs with <30 listings are pooled into `{county}_smallzips`).  
Each row also gets a commuter-corridor tag: **NYC Corridor**, **Philadelphia Corridor**, or **Other**.

In [ ]:
from src.segmentation.segment import segment_listings
seg_path = segment_listings(geo_path, cfg)
seg = read_any(seg_path)
seg['commuter_corridor'].value_counts()

## 5 · Pricing Analytics (Phase 5)

Medians, p10–p90 ranges, $/sqft, bed-bucket & property-type matrices, and a 0–100 Price Heat Index at three levels of granularity.

In [ ]:
from src.pricing.pricing_analytics import build_pricing_analytics
pa = build_pricing_analytics(seg_path, cfg)
county = read_any(pa['county'])
county.head(10)

In [ ]:
# Pricing matrix — rows = micro-markets, columns = property_type | bed_bucket
pricing_matrix = pd.read_csv(pa['pricing_matrix'])
print('Markets × cells :', pricing_matrix.shape)
pricing_matrix.head()

## 6 · Hedonic ML (Phase 6)

Linear + Ridge regression on numerical (beds, baths, sqft, lot_size, year_built) + one-hot categorical (county, municipality, property_type).  
5-fold cross-validation plus held-out test.

In [ ]:
from src.modeling.hedonic_model import train_hedonic
metrics = train_hedonic(seg_path, cfg)
print(json.dumps(metrics, indent=2, default=str))

## 7 · Clustering (Phase 7)

K-Means on (price, $/sqft) → Budget / Mid / Premium / Luxury tiers, with an elbow-method sanity check.

In [ ]:
from src.modeling.clustering import build_clusters
cl = build_clusters(seg_path, cfg)
cl['tier'].value_counts()

## 8 · Visualizations (Phase 8)

In [ ]:
from src.visualization.charts import build_all_charts
chart_paths = build_all_charts(seg_path, cfg)
from IPython.display import Image, display
for p in [chart_paths[k] for k in ['county_median_bar', 'price_per_sqft_box', 'correlation_matrix']]:
    display(Image(filename=str(p)))

## 9 · Interactive Map (Phase 9)

In [ ]:
from src.visualization.map import build_folium_map
map_path = build_folium_map(seg_path, cfg)
from IPython.display import IFrame
IFrame(src=str(map_path), width='100%', height=600)

## 10 · Summary Deck (Phase 10)

In [ ]:
from src.reporting.deck import build_deck
deck_path = build_deck(cfg)
print('Deck →', deck_path)

## Wrap-up

Outputs:
- `data/raw/nj_housing_raw_*.csv|parquet` — raw scrape  
- `data/processed/nj_housing_segmented.parquet` — analysis-ready table  
- `outputs/pricing_matrix.csv` — micro-markets × beds × type  
- `outputs/models/{linreg,ridge}.joblib` + `metrics.json`  
- `outputs/nj_housing_map.html` — Folium choropleth  
- `reports/figures/*.png` — eight static charts  
- `reports/data_quality_report.md`, `reports/model_evaluation_report.md`  
- `reports/summary_deck.pptx` — 8-slide executive deck